# Lesson 8B: Policy Gradient Methods - Practical

## Introduction

Implement policy gradient methods in PyTorch on continuous control tasks:
- Actor-Critic with neural networks
- Entropy regularization for exploration
- Compare from-scratch implementation with Stable-Baselines3
- Training on LunarLander and BipedalWalker

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from collections import deque

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Neural Network Actor-Critic

### Network Architecture
- Actor: state → hidden layers → action mean and log-std (for continuous actions)
- Critic: state → hidden layers → value
- Shared body: optional for computational efficiency

In [ ]:
class ActorNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
    
    def forward(self, state):
        features = self.net(state)
        mean = self.mean(features)
        std = torch.exp(self.log_std)
        return mean, std

class CriticNetwork(nn.Module):
    def __init__(self, state_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, state):
        return self.net(state).squeeze(-1)

print("Actor and Critic networks: defined")

## Part 2: A2C Training Loop

### Entropy Regularization

Encourage exploration via entropy penalty:
$$L = -\log \pi(a|s) (r + \gamma V(s') - V(s)) + \alpha H(\pi(\cdot|s))$$

For Gaussian policy: $H = 0.5 \log(2\pi e \sigma^2)$

In [ ]:
class A2CAgent:
    def __init__(self, state_dim: int, action_dim: int, lr: float = 1e-3, entropy_coeff: float = 0.01):
        self.actor = ActorNetwork(state_dim, action_dim).to(device)
        self.critic = CriticNetwork(state_dim).to(device)
        self.actor_opt = optim.Adam(self.actor.parameters(), lr=lr)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=lr)
        self.entropy_coeff = entropy_coeff
        self.action_dim = action_dim
    
    def select_action(self, state: np.ndarray):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            mean, std = self.actor(state_t)
            dist = torch.distributions.Normal(mean, std)
            action = dist.sample()
        return action.cpu().numpy()[0]
    
    def update(self, states, actions, rewards, next_states, dones, gamma: float = 0.99):
        states_t = torch.FloatTensor(states).to(device)
        actions_t = torch.FloatTensor(actions).to(device)
        rewards_t = torch.FloatTensor(rewards).to(device)
        next_states_t = torch.FloatTensor(next_states).to(device)
        dones_t = torch.FloatTensor(dones).to(device)
        
        # Compute TD targets and advantages
        with torch.no_grad():
            values = self.critic(states_t)
            next_values = self.critic(next_states_t)
            td_targets = rewards_t + gamma * next_values * (1 - dones_t)
            advantages = td_targets - values
        
        # Critic loss
        critic_loss = ((self.critic(states_t) - td_targets) ** 2).mean()
        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()
        
        # Actor loss with entropy regularization
        mean, std = self.actor(states_t)
        dist = torch.distributions.Normal(mean, std)
        log_probs = dist.log_prob(actions_t).sum(dim=1)
        entropy = dist.entropy().sum(dim=1).mean()
        
        actor_loss = -(log_probs * advantages).mean() - self.entropy_coeff * entropy
        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()
        
        return critic_loss.item(), actor_loss.item()

print("A2C Agent: defined")

## Part 3: Training on LunarLander

Test on continuous lunar lander environment with from-scratch A2C.

In [ ]:
# Test on LunarLander
env = gym.make('LunarLander-v2')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

agent = A2CAgent(state_dim, action_dim, entropy_coeff=0.01)

# Batch collection and updates
batch_size = 64
num_episodes = 50  # Quick training for demonstration
episode_returns = []
critic_losses = []
actor_losses = []

for episode in range(num_episodes):
    state, _ = env.reset()
    done = False
    episode_return = 0
    
    states, actions, rewards, next_states, dones = [], [], [], [], []
    
    while not done:
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        states.append(state)
        actions.append(action)
        rewards.append(reward)
        next_states.append(next_state)
        dones.append(done)
        
        episode_return += reward
        state = next_state
        
        if len(states) >= batch_size:
            c_loss, a_loss = agent.update(states, actions, rewards, next_states, dones)
            critic_losses.append(c_loss)
            actor_losses.append(a_loss)
            states, actions, rewards, next_states, dones = [], [], [], [], []
    
    episode_returns.append(episode_return)
    if (episode + 1) % 10 == 0:
        print(f"Episode {episode + 1}: Return = {episode_return:.2f}")

env.close()
print(f"\nTraining complete. Final avg return: {np.mean(episode_returns[-10:]):.2f}")

## Part 4: Stable-Baselines3 Reproduction

Compare with SB3 A2C for validation.

In [ ]:
try:
    from stable_baselines3 import A2C
    
    env = gym.make('LunarLander-v2')
    model = A2C('MlpPolicy', env, verbose=0)
    
    # Quick training
    model.learn(total_timesteps=5000)
    
    # Evaluate
    returns_sb3 = []
    for _ in range(5):
        obs, _ = env.reset()
        done = False
        ret = 0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ret += reward
        returns_sb3.append(ret)
    
    env.close()
    print(f"\nStable-Baselines3 A2C avg return: {np.mean(returns_sb3):.2f}")
except ImportError:
    print("Stable-Baselines3 not available. Install: pip install stable-baselines3")

## Key Takeaways

1. **Policy networks** parameterize continuous action distributions
2. **Entropy regularization** encourages exploration
3. **Actor-Critic** with neural networks is practical and stable
4. **A2C** batching improves sample efficiency
5. **Stable-Baselines3** provides production implementations

Next: Trust region methods (TRPO, PPO) for even more stable optimization.